# NumPy 极简教程

本教程从基础到进阶，覆盖图像处理与神经网络计算中常用的 NumPy 核心技能。

**目录**
1. 导入 NumPy 库
2. 创建数组
3. 数组属性
4. 数组索引与切片
5. 数组运算与广播
6. 数组变形、转置与 einsum
7. 常用数学函数
8. 随机数生成

## 1. 导入 NumPy 库

In [1]:
import numpy as np

# 打印版本号，确认安装成功
print("NumPy version:", np.__version__)

NumPy version: 1.26.4


## 2. 创建数组

NumPy 数组（`ndarray`）是连续存储的多维数值容器，比 Python 列表快得多。

In [2]:
# 从列表创建一维数组
a = np.array([1, 2, 3, 4, 5])
print("一维数组:", a)

# 从嵌套列表创建二维数组
b = np.array([[1, 2, 3], [4, 5, 6]])
print("二维数组:\n", b)

# 常用快捷函数
print("\nnp.zeros((2, 3)):\n", np.zeros((2, 3)))          # 全零数组
print("np.ones((2, 3)):\n", np.ones((2, 3)))              # 全一数组
print("np.arange(0, 10, 2):", np.arange(0, 10, 2))       # 等差序列（步长 2）
print("np.linspace(0, 1, 5):", np.linspace(0, 1, 5))     # 等间隔 5 个点

一维数组: [1 2 3 4 5]
二维数组:
 [[1 2 3]
 [4 5 6]]

np.zeros((2, 3)):
 [[0. 0. 0.]
 [0. 0. 0.]]
np.ones((2, 3)):
 [[1. 1. 1.]
 [1. 1. 1.]]
np.arange(0, 10, 2): [0 2 4 6 8]
np.linspace(0, 1, 5): [0.   0.25 0.5  0.75 1.  ]


## 3. 数组属性

每个 `ndarray` 都携带若干描述自身结构的属性。

In [3]:
img = np.zeros((32, 32, 3))           # 模拟一张 32x32 的 RGB 图像

print("维度数  ndim :", img.ndim)     # 3
print("形状    shape:", img.shape)    # (32, 32, 3)
print("元素总数 size :", img.size)    # 3072
print("数据类型 dtype:", img.dtype)   # float64

# astype 转换数据类型（图像分类中常将像素值从 uint8 转为 float32）
img_f32 = img.astype(np.float32)
print("转换后  dtype:", img_f32.dtype)

维度数  ndim : 3
形状    shape: (32, 32, 3)
元素总数 size : 3072
数据类型 dtype: float64
转换后  dtype: float32


## 4. 数组索引与切片

NumPy 支持整数索引、切片、布尔索引和花式索引（Fancy Indexing）。

In [4]:
arr = np.arange(10)          # [0 1 2 3 4 5 6 7 8 9]
print("原数组:", arr)

# 整数索引
print("arr[3]  =", arr[3])          # 3

# 切片  arr[start:stop:step]
print("arr[2:7]   =", arr[2:7])     # [2 3 4 5 6]
print("arr[::2]   =", arr[::2])     # [0 2 4 6 8]  隔一取一
print("arr[::-1]  =", arr[::-1])    # 反转

# 二维切片
M = np.arange(12).reshape(3, 4)
print("\n二维数组 M:\n", M)
print("第 1 行:", M[1])             # 整行
print("第 2 列:", M[:, 2])          # 整列
print("子矩阵:\n", M[0:2, 1:3])    # 取左上 2x2 子块

# 布尔索引（条件筛选）
print("\n大于 5 的元素:", arr[arr > 5])

# 花式索引（Fancy Indexing）：用整数数组指定下标，批量提取多个元素
class_labels = np.array([3, 1, 4, 1, 5, 9, 2, 6])
sel_idx = np.array([0, 3, 6])       # 选取第 0、3、6 个位置
print("选取的标签:", class_labels[sel_idx])   # [3 1 2]

# 二维花式索引：同时指定行和列（常用于从预测矩阵中取特定类别的得分）
scores = np.array([[0.1, 0.7, 0.2],
                   [0.8, 0.1, 0.1],
                   [0.3, 0.4, 0.3]])
row_idx = np.array([0, 1, 2])
col_idx = np.array([1, 0, 1])      # 每行取的列
print("各行指定列的得分:", scores[row_idx, col_idx])   # [0.7 0.8 0.4]

原数组: [0 1 2 3 4 5 6 7 8 9]
arr[3]  = 3
arr[2:7]   = [2 3 4 5 6]
arr[::2]   = [0 2 4 6 8]
arr[::-1]  = [9 8 7 6 5 4 3 2 1 0]

二维数组 M:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
第 1 行: [4 5 6 7]
第 2 列: [ 2  6 10]
子矩阵:
 [[1 2]
 [5 6]]

大于 5 的元素: [6 7 8 9]
选取的标签: [3 1 2]
各行指定列的得分: [0.7 0.8 0.4]


## 5. 数组运算与广播

### 逐元素运算
对两个形状相同的数组，运算逐元素执行：
$$c_i = a_i \odot b_i$$

### 广播机制（Broadcasting）
形状不同的数组按以下规则自动扩展：
- 从**右侧**对齐维度
- 尺寸为 1 的维度自动复制以匹配对方

例如，将 $(N, D)$ 矩阵的每一行减去一个形状为 $(D,)$ 的向量：
$$M'_{ij} = M_{ij} - v_j$$

对四维图像批次 $(N, C, H, W)$ 按通道归一化时，均值和标准差形状为 $(1, C, 1, 1)$，一条语句即可完成所有像素的标准化。

In [5]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# 逐元素算术运算
print("a + b  =", a + b)
print("a - b  =", a - b)
print("a * b  =", a * b)      # 逐元素乘（非矩阵乘）
print("a ** 2 =", a ** 2)     # 逐元素平方

# ------ 广播示例 1：二维矩阵减一维向量 ------
rng = np.random.default_rng(42)
X = rng.integers(0, 256, size=(5, 4)).astype(np.float64)
col_mean = X.mean(axis=0)      # 列均值，shape (4,)
X_centered = X - col_mean      # (5,4) - (4,) → 自动广播

print("\nX:\n", X.round(1))
print("col_mean:", col_mean.round(1))
print("X_centered:\n", X_centered.round(1))

# ------ 广播示例 2：四维图像批次的通道级归一化 ------
# batch: (N=8, C=3, H=32, W=32) 模拟 8 张 RGB 图像
batch = rng.integers(0, 256, size=(8, 3, 32, 32)).astype(np.float32)

# 沿 N/H/W 三个轴求均值和标准差，保留 C 轴，keepdims 维持形状便于广播
ch_mean = batch.mean(axis=(0, 2, 3), keepdims=True)   # (1, 3, 1, 1)
ch_std  = batch.std(axis=(0, 2, 3),  keepdims=True) + 1e-8

batch_norm = (batch - ch_mean) / ch_std               # (8,3,32,32)，全部广播

print("\nbatch_norm shape:", batch_norm.shape)
print("各通道归一化后均值（应接近 0）:", batch_norm.mean(axis=(0, 2, 3)).round(6))
print("各通道归一化后标准差（应接近 1）:", batch_norm.std(axis=(0, 2, 3)).round(4))

# 比较运算（返回布尔数组）
print("\na > 2 :", a > 2)

a + b  = [5. 7. 9.]
a - b  = [-3. -3. -3.]
a * b  = [ 4. 10. 18.]
a ** 2 = [1. 4. 9.]

X:
 [[ 22. 198. 167. 112.]
 [110. 219.  22. 178.]
 [ 51.  24. 134. 249.]
 [188. 194. 183. 201.]
 [131.  32. 214. 115.]]
col_mean: [100.4 133.4 144.  171. ]
X_centered:
 [[ -78.4   64.6   23.   -59. ]
 [   9.6   85.6 -122.     7. ]
 [ -49.4 -109.4  -10.    78. ]
 [  87.6   60.6   39.    30. ]
 [  30.6 -101.4   70.   -56. ]]

batch_norm shape: (8, 3, 32, 32)
各通道归一化后均值（应接近 0）: [ 0. -0. -0.]
各通道归一化后标准差（应接近 1）: [1. 1. 1.]

a > 2 : [False False  True]


## 6. 数组变形、转置与 einsum

- `reshape(new_shape)` — 改变形状，元素总数不变
- `flatten()` — 展平为一维
- `.T` / `transpose()` — 转置 $A^T$，常用于矩阵乘时对齐维度
- `np.einsum(subscripts, ...)` — Einstein 求和，用简洁的下标符号表达任意张量运算

In [6]:
x = np.arange(12)
print("原数组:", x)

# reshape：1D → 3x4 矩阵，-1 让 NumPy 自动推断该维度
A = x.reshape(3, 4)
B = x.reshape(2, -1)          # 2 行，列数自动算出为 6
print("reshape(3,4):\n", A)
print("reshape(2,-1):\n", B)

# 展平 / 转置
print("flatten:", A.flatten())
print("A.T shape:", A.T.shape)   # (4, 3)

# ---- np.einsum ----
rng = np.random.default_rng(42)
P = rng.random((3, 4))
Q = rng.random((4, 5))

# 矩阵乘：等价于 P @ Q
R1 = np.einsum('ij,jk->ik', P, Q)
R2 = P @ Q
print("\neinsum 矩阵乘 vs @，最大误差:", np.abs(R1 - R2).max())

# 逐行点积（每行自身内积 → 行的 L2² 范数）
row_sq = np.einsum('ij,ij->i', P, P)           # shape (3,)
print("逐行点积（einsum）:", row_sq.round(4))
print("等价的 (P**2).sum(axis=1):", (P**2).sum(axis=1).round(4))

# 批量矩阵乘：(B, M, K) × (B, K, N) → (B, M, N)
rng2 = np.random.default_rng(7)
batch_A = rng2.random((4, 3, 5))   # 4 个 3×5 矩阵
batch_B = rng2.random((4, 5, 2))   # 4 个 5×2 矩阵
batch_C = np.einsum('bij,bjk->bik', batch_A, batch_B)   # (4, 3, 2)
print("\n批量矩阵乘 shape:", batch_C.shape)

原数组: [ 0  1  2  3  4  5  6  7  8  9 10 11]
reshape(3,4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
reshape(2,-1):
 [[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
flatten: [ 0  1  2  3  4  5  6  7  8  9 10 11]
A.T shape: (4, 3)

einsum 矩阵乘 vs @，最大误差: 0.0
逐行点积（einsum）: [2.0151 2.1579 1.2156]
等价的 (P**2).sum(axis=1): [2.0151 2.1579 1.2156]

批量矩阵乘 shape: (4, 3, 2)


## 7. 常用数学函数

### 7.1 聚合函数与 axis 参数

`axis=0` 沿行方向归约（列保留），`axis=1` 沿列方向归约（行保留）：

$$\text{sum}(A, \text{axis}=1)_i = \sum_j A_{ij}$$

`keepdims=True` 保持原有维度数，方便后续广播；不加则会"压缩"掉该轴。

### 7.2 核心函数速查

| 函数 | 说明 |
|------|------|
| `np.dot(a, b)` / `A @ B` | 向量点积 / 矩阵乘 |
| `np.linalg.norm(x, axis=1)` | 计算每行向量的 L2 范数 |
| `np.sqrt(x)` | 逐元素开平方（计算欧氏距离） |
| `np.exp(x)` / `np.log(x)` | 逐元素指数 / 自然对数（Softmax、交叉熵） |
| `np.clip(x, a_min, a_max)` | 值域钳制，防止 `log(0)` 或负数开方 |
| `np.argmax(x, axis=1)` | 每行最大值的列索引（取分类预测结果） |
| `np.allclose(a, b)` | 判断两数组是否在容差内近似相等（验证两种实现等价） |

In [7]:
# ---- 聚合函数与 axis 参数 ----
M = np.array([[1, 2, 3],
              [4, 5, 6]])

print("sum (全部):", M.sum())           # 21
print("sum axis=0:", M.sum(axis=0))    # 按列求和 → [5 7 9]
print("sum axis=1:", M.sum(axis=1))    # 按行求和 → [6 15]

# keepdims=True 保持维度，便于广播
sq = (M ** 2).sum(axis=1, keepdims=True)
print("逐行平方和 (keepdims=True):\n", sq)   # shape (2,1)

print("\nmean:", M.mean())
print("std :", M.std())
print("min :", M.min(),  "  max:", M.max())

sum (全部): 21
sum axis=0: [5 7 9]
sum axis=1: [ 6 15]
逐行平方和 (keepdims=True):
 [[14]
 [77]]

mean: 3.5
std : 1.707825127659933
min : 1   max: 6


In [8]:
# ---- np.dot / 矩阵乘 @ ----
v = np.array([1.0, 2.0, 3.0])
w = np.array([4.0, 5.0, 6.0])
print("点积 np.dot(v,w):", np.dot(v, w))   # 1*4+2*5+3*6 = 32

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("矩阵乘 A @ B:\n", A @ B)

# ---- np.sqrt 与 np.clip ----
vals = np.array([-1.0, 0.0, 4.0, 9.0])
clipped = np.clip(vals, 0, None)        # 将负数钳制为 0（避免开方产生 NaN）
print("\nclip 后:", clipped)
print("sqrt  :", np.sqrt(clipped))

# ---- np.exp / np.log（Softmax 与交叉熵的基础操作）----
logits = np.array([2.0, 1.0, 0.1])
exp_z  = np.exp(logits - logits.max())  # 数值稳定：先减最大值
probs  = exp_z / exp_z.sum()            # Softmax 概率，和为 1
print("\nlogits:", logits)
print("Softmax probs:", probs.round(4))
print("log(probs):", np.log(probs).round(4))   # 交叉熵用到 -log(p)

# ---- np.argmax：取每行最大值的列索引（分类预测）----
scores = np.array([[0.1, 0.9, 0.0],
                   [0.7, 0.2, 0.1],
                   [0.3, 0.3, 0.4]])
preds = np.argmax(scores, axis=1)
print("\n原始分数:\n", scores)
print("预测类别（argmax）:", preds)   # [1 0 2]

点积 np.dot(v,w): 32.0
矩阵乘 A @ B:
 [[19 22]
 [43 50]]

clip 后: [0. 0. 4. 9.]
sqrt  : [0. 0. 2. 3.]

logits: [2.  1.  0.1]
Softmax probs: [0.659  0.2424 0.0986]
log(probs): [-0.417 -1.417 -2.317]

原始分数:
 [[0.1 0.9 0. ]
 [0.7 0.2 0.1]
 [0.3 0.3 0.4]]
预测类别（argmax）: [1 0 2]


### np.allclose — 数值等价验证

浮点运算因精度原因常产生微小误差，`np.allclose(a, b, rtol, atol)` 按下式判断：
$$|a_i - b_i| \leq \text{atol} + \text{rtol} \times |b_i|$$

常用场景：验证两种实现（如手写 vs 库函数）的结果是否一致。

In [9]:
rng = np.random.default_rng(42)
X = rng.random((100, 50)).astype(np.float32)

# 两种计算 L2 范数的方式
norms_linalg = np.linalg.norm(X, axis=1)
norms_manual = np.sqrt((X ** 2).sum(axis=1))

print("linalg.norm 前 5 行:", norms_linalg[:5].round(6))
print("手工开方    前 5 行:", norms_manual[:5].round(6))

# allclose 验证两者等价（容许浮点误差）
print("\nnp.allclose:", np.allclose(norms_linalg, norms_manual, atol=1e-5))

# any：检查是否存在不满足条件的值
max_diff = np.abs(norms_linalg - norms_manual).max()
print("最大绝对误差:", max_diff)
print("误差超出 1e-4:", np.any(np.abs(norms_linalg - norms_manual) > 1e-4))

# 对比两种矩阵乘的等价性（einsum vs @）
A = rng.random((5, 8)).astype(np.float32)
B = rng.random((8, 3)).astype(np.float32)
print("\neinsum 与 @ 等价:", np.allclose(np.einsum('ij,jk->ik', A, B), A @ B))

linalg.norm 前 5 行: [4.2642   3.594159 3.918381 4.264574 4.130355]
手工开方    前 5 行: [4.2642   3.594159 3.918381 4.264574 4.130355]

np.allclose: True
最大绝对误差: 0.0
误差超出 1e-4: False

einsum 与 @ 等价: True


## 8. 随机数生成

推荐使用新式 API `np.random.default_rng(seed)` 代替旧式 `np.random.seed()` + `np.random.xxx`，它是线程安全的，且更易于复现实验结果。

In [10]:
rng = np.random.default_rng(seed=42)   # 固定随机种子，保证结果可复现

# [0,1) 均匀分布
print("uniform (3,4):\n", rng.random((3, 4)).round(3))

# 标准正态分布  N(0,1)
print("\nnormal (2,3):\n", rng.standard_normal((2, 3)).round(3))

# 整数随机（用于数据集采样）
idx = rng.integers(0, 50000, size=500)   # 从 5 万样本中随机选 500 个下标
print("\nsampled indices[:10]:", idx[:10])

# 旧式 API（仍可用，了解即可）
np.random.seed(42)
print("\n旧式 rand:", np.random.rand(3).round(3))
print("旧式 randint:", np.random.randint(0, 10, size=5))

uniform (3,4):
 [[0.774 0.439 0.859 0.697]
 [0.094 0.976 0.761 0.786]
 [0.128 0.45  0.371 0.927]]

normal (2,3):
 [[ 0.066  1.127  0.468]
 [-0.859  0.369 -0.959]]

sampled indices[:10]: [42914 41381 13837 31583  8261 37904 35026 17726  3396 48534]

旧式 rand: [0.375 0.951 0.732]
旧式 randint: [4 6 9 2 6]
